In [ ]:
from pyDOE import lhs
from smt.sampling_methods import LHS
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import cv2
import json
import seaborn as sns

import asyncio
import websockets
import base64
from PIL import Image as PILImage, ImageDraw
from IPython.display import display, Image as IPythonImage
from ipywidgets import interact, IntSlider, fixed

In [ ]:
async def capture_photo():
    uri = "ws://172.28.236.86:8765"
    
    async with websockets.connect(uri, max_size=5_000_000) as websocket:
        # send command to capture
        await websocket.send("capture")
        
        # wait for response
        response = await websocket.recv()
        
        try:
            # decode from Base64
            img_data = base64.b64decode(response)
            
            display(IPythonImage(data=img_data, format='jpg'))
            print("Image captured")

            with open("captured_image.jpg", "wb") as f:
                f.write(img_data)
            print("Image saved as captured_image.jpg")

        except Exception as e:
            print("Response:", response)
            print("Exception:", e)

result = await capture_photo()
print(result)

In [ ]:
image_path = 'captured_image.jpg'

# parameters for bilateral filtering
diameter = 5
sigmaColor = 75
sigmaSpace = 75

image_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
filtered_image = cv2.bilateralFilter(image_gray, diameter, sigmaColor, sigmaSpace)
image_rgb = cv2.imread(image_path)

rows = 8
cols = 12

# define coordinates for top-left and bottom-right grid points
top_left = (265, 278)
bottom_right = (2989, 2006)

x_step = (bottom_right[0] - top_left[0]) / (cols - 1)
y_step = (bottom_right[1] - top_left[1]) / (rows - 1)

expected_grid = []
rgb_values = []

for row in range(rows):
    for col in range(cols):
        x = int(top_left[0] + col * x_step)
        y = int(top_left[1] + row * y_step)
        expected_grid.append((x, y))
        
        # image[row, col] → image[y, x]
        b, g, r = image_rgb[y, x]  
        rgb_values.append([r, g, b]) 

output_grid = image_rgb.copy()

# draw circle
for (x, y) in expected_grid:
    cv2.circle(output_grid, (x, y), 100, (255, 0, 0), 2)

output_grid_rgb = cv2.cvtColor(output_grid, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(output_grid_rgb)
plt.axis("off")
plt.title("Contour Detection")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# make sure to have exact number of rows and columns
if len(rgb_values) < rows * cols:
    rgb_values += [[0, 0, 0]] * (rows * cols - len(rgb_values))

for row in range(rows):
    for col in range(cols):
        index = row * cols + col
        rgb = rgb_values[index]
        # normalize RGB values to [0, 1]
        color = [val / 255 for val in rgb]
        # note the y-coordinate is inverted for display.
        circle = plt.Circle((col + 0.5, rows - row - 0.5), 0.4, facecolor=color, 
                            edgecolor='black', linewidth=1)
        ax.add_patch(circle)

print('1st well:', rgb_values[84])
print('2nd well:', rgb_values[72])

ax.set_xlim(0, cols)
ax.set_ylim(0, rows)
ax.set_aspect('equal')
ax.axis('off') 
plt.title('8x12 Grid of Colors')
plt.show()